# DESC ELAsTiCC2 — SALT3 fit: light curves, error ellipses and correlation matrix

- **author** : Sylvie Dagoret-Campagne
- **affiliation** : IJCLab/IN2P3/CNRS — Université Paris-Saclay
- **creation date** : 2026-05-08
- **based on** : `02_sncosmo/03_elasticc2_fitsalt3_lightcurves.ipynb`

## Purpose

For each fitted SNIa event this notebook produces **three figures**:

1. **Multi-band light curve + SALT3 model** (identical to notebook 03) — flux data
   points per band with error bars and the fitted SALT3 model curves.
2. **Pairwise error ellipses (corner plot)** — for the 4 free parameters
   `(t0, x0, x1, c)`, every pair $(p_i, p_j)$ is shown as a 2D confidence
   ellipse at **1σ** and **2σ** computed analytically from the 4×4 covariance
   matrix returned by `sncosmo.fit_lc`.  The diagonal panels show the 1D Gaussian
   marginal with the fitted value ± 1σ.
3. **Parameter correlation matrix** — heatmap of the Pearson correlation matrix
   $\rho_{ij} = C_{ij}/\sqrt{C_{ii}\,C_{jj}}$ with annotated values.

## Error ellipse — mathematical background

Given the 2×2 sub-covariance matrix
$\mathbf{C}_{2\times2} = \begin{pmatrix} \sigma_i^2 & \rho\,\sigma_i\sigma_j \\
                                           \rho\,\sigma_i\sigma_j & \sigma_j^2 \end{pmatrix}$,
the boundary of the joint **$n\sigma$ confidence ellipse** satisfies
$\mathbf{\delta}^T\,\mathbf{C}^{-1}\,\mathbf{\delta} = \Delta\chi^2$
where $\Delta\chi^2 = 2.30$ (1σ, 2 d.o.f.) and $\Delta\chi^2 = 6.18$ (2σ).

This is equivalent to an ellipse whose semi-axes are the square roots of the
eigenvalues of $\mathbf{C}_{2\times2}$ scaled by $\sqrt{\Delta\chi^2}$, rotated
by the eigenvectors.

### References
- Kenworthy et al. 2021 — SALT3: https://doi.org/10.3847/1538-4357/ac30d8
- sncosmo documentation: https://sncosmo.readthedocs.io/en/stable/index.html
- ELAsTiCC2 dataset: DESC TD public data
- Notebook `02_sncosmo/03_elasticc2_fitsalt3_lightcurves.ipynb` — SALT3 baseline


## 0 · Imports

In [ ]:
%matplotlib inline

import sys
import os
import math
import pathlib
import logging
import warnings

import numpy as np
import pandas as pd
import astropy.table
import matplotlib
import matplotlib.patches as mpatches
from matplotlib import pyplot as plt
from matplotlib.patches import Ellipse

import sncosmo

# ── local library ──────────────────────────────────────────────────────────────
libdir = pathlib.Path(os.getcwd()).parent.parent / "lib_elasticc2"
sys.path.insert(0, str(libdir))
from transcientslightcurves import elasticc2_snana_reader

# ── logging ────────────────────────────────────────────────────────────────────
_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler(sys.stderr)
    _logger.addHandler(_logout)
    _logout.setFormatter(logging.Formatter(
        '[%(asctime)s - %(levelname)s] - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
_logger.setLevel(logging.INFO)
_logger.info("Imports done.")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
try:
    import ipympl  # noqa: F401
    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline")
    print("Install with:  pip install ipympl")

## 1 · Parameters

In [ ]:
# ── Data selection ─────────────────────────────────────────────────────────────
OBJ_CLASS      = 'SNIa-SALT3'   # SNANA class label
Z_MIN          = 0.1            # redshift lower bound
Z_MAX          = 1.0            # redshift upper bound
FILE_NUM       = 1              # PHOT file index (1–40; None = all)
MIN_DETECTIONS = 8              # minimum detected points per object
DETECTED_ONLY  = True           # use only detected points (PHOTFLAG & photflag_detect)
N_CURVES       = 6              # number of events to display in full detail
RANDOM_SEED    = 42

# ── SALT3 model ────────────────────────────────────────────────────────────────
SALT3_SOURCE   = 'salt3'        # sncosmo source name (Kenworthy et al. 2021)
SALT3_VERSION  = '2.0'          # version used in the SNANA simulation
ZP             = 31.4           # zero-point (AB system)
ZPSYS          = 'ab'

BANDS       = ['u', 'g', 'r', 'i', 'z', 'y']
BAND_PREFIX = 'lsst'

# ── Data path ──────────────────────────────────────────────────────────────────
DATA_DIR   = "/Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
DIR_PREFIX = "ELASTICC2_TRAIN_02_"

# ── Colours per band ───────────────────────────────────────────────────────────
BAND_COLORS = {
    'u': '#cc0ccc',
    'g': '#00cc44',
    'r': '#cc0000',
    'i': '#ff4400',
    'z': '#886600',
    'y': '#442200'
}

# ── Free parameters fitted by sncosmo ─────────────────────────────────────────
# z is fixed to truth; the 4 free parameters below enter the covariance matrix.
VPARAM_NAMES = ['t0', 'x0', 'x1', 'c']

# Pretty LaTeX labels for plots
PARAM_LABELS = {
    't0' : r'$t_0$ [days]',
    'x0' : r'$x_0$',
    'x1' : r'$x_1$',
    'c'  : r'$c$',
}

# Δχ² thresholds for 1σ and 2σ confidence ellipses (2 d.o.f.)
DCHI2_1SIGMA = 2.30    # 68.3 % — 1σ for 2 parameters jointly
DCHI2_2SIGMA = 6.18    # 95.4 % — 2σ for 2 parameters jointly

rng = np.random.default_rng(seed=RANDOM_SEED)
print(f"SALT3 source : {SALT3_SOURCE}  v{SALT3_VERSION}")
print(f"Free params  : {VPARAM_NAMES}")
print(f"N_CURVES     : {N_CURVES}")

## 2 · Load ELAsTiCC2 data

In [ ]:
esr = elasticc2_snana_reader(DATA_DIR, dir_prefix=DIR_PREFIX)

_logger.info(f"Loading HEAD for {OBJ_CLASS}...")
head      = esr.get_head(OBJ_CLASS, return_format='pandas')
_logger.info("Loading truth...")
truth     = esr.get_object_truth(OBJ_CLASS, return_format='pandas')
_logger.info(f"Loading light curves (file_num={FILE_NUM})...")
all_ltcvs = esr.get_all_ltcvs(OBJ_CLASS, file_num=FILE_NUM, return_format='pandas')
_logger.info("Done.")

print(f"{all_ltcvs['SNID'].nunique()} objects loaded.")

In [ ]:
# ── Filter: redshift range + minimum detections ───────────────────────────────
detcounts = (
    all_ltcvs[(all_ltcvs['PHOTFLAG'] & esr.photflag_detect) != 0]
    .groupby('SNID').agg('count')['MJD']
    .reset_index()
    .rename({'MJD': 'ndetect'}, axis=1)
)
truth_counts = truth.join(detcounts.set_index('SNID'), on='SNID', how='inner')
subset = truth_counts[
    (truth_counts['ZCMB'] >= Z_MIN) &
    (truth_counts['ZCMB'] <  Z_MAX) &
    (truth_counts['ndetect'] >= MIN_DETECTIONS)
].copy()
print(f"{len(subset)} objects pass selection.")

## 3 · Helper functions

### 3.1 · ELAsTiCC2 DataFrame → sncosmo astropy.Table

In [ ]:
def make_sncosmo_table(ltcv_df: pd.DataFrame,
                       detected_only: bool = True,
                       photflag_detect: int = None,
                       zp: float = ZP,
                       zpsys: str = ZPSYS,
                       band_prefix: str = BAND_PREFIX) -> astropy.table.Table:
    """Convert an ELAsTiCC2 light-curve DataFrame to an astropy.Table
    suitable for sncosmo.fit_lc."""
    df = ltcv_df.copy()
    if detected_only and photflag_detect is not None:
        df = df[(df['PHOTFLAG'] & photflag_detect) != 0]
    df = df[df['FLUXCALERR'] > 0].copy()
    df['BAND'] = df['BAND'].str.strip().str.lower()
    df['band_sncosmo'] = band_prefix + df['BAND']
    return astropy.table.Table({
        'time'    : df['MJD'].values.astype(float),
        'band'    : df['band_sncosmo'].values,
        'flux'    : df['FLUXCAL'].values.astype(float),
        'fluxerr' : df['FLUXCALERR'].values.astype(float),
        'zp'      : np.full(len(df), zp, dtype=float),
        'zpsys'   : np.full(len(df), zpsys),
    })

print("make_sncosmo_table ready.")

### 3.2 · SALT3 fitter

In [ ]:
def fit_salt3_event(ltcv_df: pd.DataFrame,
                    z_true: float,
                    fit_z: bool = False,
                    photflag_detect: int = None) -> dict:
    """Fit a single ELAsTiCC2 SNIa event with SALT3.

    Returns a dict containing the fitted model, the full 4×4 covariance
    matrix C (over the free parameters t0, x0, x1, c), the derived
    correlation matrix, and 1-sigma marginal uncertainties.
    """
    try:
        obs = make_sncosmo_table(ltcv_df, detected_only=DETECTED_ONLY,
                                 photflag_detect=photflag_detect)
    except Exception as e:
        return {'success': False, 'message': f'Table creation failed: {e}'}

    if len(obs) < 5:
        return {'success': False, 'message': 'Not enough data points'}

    model = sncosmo.Model(
        source=sncosmo.get_source(SALT3_SOURCE, version=SALT3_VERSION)
    )
    t0_guess = float(obs['time'][np.argmax(obs['flux'])])
    model.set(z=z_true, t0=t0_guess, x0=1e-4, x1=0.0, c=0.0)

    vparam_names = list(VPARAM_NAMES)   # ['t0','x0','x1','c']
    bounds = {
        't0' : (t0_guess - 30.0, t0_guess + 30.0),
        'x0' : (1e-8, 1.0),
        'x1' : (-5.0, 5.0),
        'c'  : (-0.5, 0.5),
    }
    if fit_z:
        vparam_names = ['z'] + vparam_names
        bounds['z'] = (max(z_true - 0.1, 0.001), z_true + 0.1)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            result, fitted_model = sncosmo.fit_lc(
                obs, model,
                vparam_names=vparam_names,
                bounds=bounds,
                minsnr=0.0,
                warn=False
            )
    except Exception as e:
        return {'success': False, 'message': f'Fit failed: {e}', 'table': obs}

    chi2     = float(result.chisq)
    ndof     = int(result.ndof)
    chi2_red = chi2 / max(ndof, 1)

    # ── Covariance and correlation matrices ───────────────────────────────────
    vcov = result.covariance   # ndarray shape (n_vparams, n_vparams) or None
    param_errors = {}
    corr_matrix  = None

    if vcov is not None:
        diag = np.diag(vcov)
        sigmas = np.sqrt(np.maximum(diag, 0.0))
        for i, pname in enumerate(vparam_names):
            param_errors[pname] = float(sigmas[i])
        # Correlation matrix: rho_ij = C_ij / (sigma_i * sigma_j)
        # Guard against zero-variance (degenerate fit)
        outer = np.outer(sigmas, sigmas)
        outer[outer == 0] = np.nan
        corr_matrix = vcov / outer
        np.fill_diagonal(corr_matrix, 1.0)

    return {
        'success'        : True,
        'result'         : result,
        'fitted_model'   : fitted_model,
        'table'          : obs,
        'vparam_names'   : vparam_names,
        'z'              : float(fitted_model['z']),
        't0'             : float(fitted_model['t0']),
        'x0'             : float(fitted_model['x0']),
        'x1'             : float(fitted_model['x1']),
        'c'              : float(fitted_model['c']),
        'chi2'           : chi2,
        'ndof'           : ndof,
        'chi2_red'       : chi2_red,
        'vcov'           : vcov,          # full covariance matrix
        'corr_matrix'    : corr_matrix,   # correlation matrix
        'param_errors'   : param_errors,  # 1-sigma marginal errors
        'message'        : result.message,
    }

print("SALT3 fitter ready.")

### 3.3 · Error ellipse drawing

Given a 2×2 sub-covariance matrix, the $n\sigma$ confidence ellipse is the locus
of points satisfying
$\mathbf{\delta}^T\,\mathbf{C}_{2\times2}^{-1}\,\mathbf{\delta} = \Delta\chi^2(n\sigma)$.

We construct it by:
1. Eigendecomposition of $\mathbf{C}_{2\times2}$ → eigenvalues $\lambda_1 \geq \lambda_2$
   and rotation angle $\theta$.
2. Semi-axes $a = \sqrt{\lambda_1 \cdot \Delta\chi^2}$,
   $b = \sqrt{\lambda_2 \cdot \Delta\chi^2}$.
3. `matplotlib.patches.Ellipse` centred on the best-fit values, rotated by $\theta$.

In [ ]:
def cov2x2_to_ellipse_params(cov2: np.ndarray, dchi2: float):
    """Return (width, height, angle_deg) of a confidence ellipse.

    Parameters
    ----------
    cov2   : 2×2 covariance sub-matrix for parameters (p_i, p_j)
    dchi2  : Δχ² threshold (2.30 for 1σ, 6.18 for 2σ with 2 d.o.f.)

    Returns
    -------
    width, height : full axes (2 × semi-axis) of the ellipse
    angle_deg     : rotation angle in degrees (from x-axis to the major axis)
    """
    eigvals, eigvecs = np.linalg.eigh(cov2)   # eigenvalues sorted ascending
    # Semi-axes scaled by sqrt(dchi2)
    # eigh returns eigenvalues in ascending order; major axis = largest eigenvalue
    a = np.sqrt(max(eigvals[1], 0.0) * dchi2)   # semi-major
    b = np.sqrt(max(eigvals[0], 0.0) * dchi2)   # semi-minor
    # Rotation: angle of the major-axis eigenvector w.r.t. x-axis
    angle_rad = np.arctan2(eigvecs[1, 1], eigvecs[0, 1])
    return 2.0 * a, 2.0 * b, np.degrees(angle_rad)


def add_error_ellipse(ax, cx: float, cy: float, cov2: np.ndarray,
                      color: str = 'C0', alpha1: float = 0.35,
                      alpha2: float = 0.15, zorder: int = 2):
    """Draw 1σ (filled) and 2σ (outline) error ellipses on ax.

    Parameters
    ----------
    ax            : matplotlib Axes
    cx, cy        : centre coordinates (best-fit parameter values)
    cov2          : 2×2 covariance sub-matrix
    color         : ellipse colour
    alpha1/alpha2 : fill transparency for 1σ / 2σ ellipses
    """
    # Catch degenerate covariance (zeros on diagonal)
    if np.any(np.diag(cov2) <= 0) or not np.all(np.isfinite(cov2)):
        return

    for dchi2, alpha, lw, ls in [
        (DCHI2_1SIGMA, alpha1, 1.5, '-'),
        (DCHI2_2SIGMA, alpha2, 1.0, '--'),
    ]:
        w, h, angle = cov2x2_to_ellipse_params(cov2, dchi2)
        ell = Ellipse(
            xy=(cx, cy), width=w, height=h, angle=angle,
            facecolor=color, alpha=alpha,
            edgecolor=color, linewidth=lw, linestyle=ls,
            zorder=zorder
        )
        ax.add_patch(ell)


print("Error ellipse helpers ready.")

### 3.4 · Plotting functions

In [ ]:
# ── Figure 1 : multi-band light curve ─────────────────────────────────────────

def plot_lightcurve(ax, snid: int, res: dict, z_true: float):
    """Plot multi-band light curve + SALT3 model on a single Axes."""
    if not res.get('success'):
        ax.set_title(f"SNID {snid}\nFit failed", fontsize=9, color='red')
        return

    obs          = res['table']
    fitted_model = res['fitted_model']
    t0           = res['t0']
    pe           = res.get('param_errors', {})

    t_min = float(obs['time'].min())
    t_max = float(obs['time'].max())
    t_dense = np.linspace(t_min - 10, t_max + 10, 400)

    for b in BANDS:
        bname = BAND_PREFIX + b
        color = BAND_COLORS[b]
        mask  = np.array(obs['band']) == bname
        if mask.sum() > 0:
            ax.errorbar(
                obs['time'][mask] - t0, obs['flux'][mask],
                yerr=obs['fluxerr'][mask],
                color=color, ls='None', marker='o', ms=4,
                capsize=2, label=b, zorder=3
            )
        try:
            f_model = fitted_model.bandflux(bname, t_dense, zp=ZP, zpsys=ZPSYS)
            valid   = np.isfinite(f_model)
            if valid.sum() > 1:
                ax.plot(t_dense[valid] - t0, f_model[valid],
                        color=color, lw=1.5, zorder=2)
        except Exception:
            pass

    ax.axhline(0.0, color='k', lw=0.5, ls='--')
    ax.set_xlabel(r'$t - t_0$ [days]', fontsize=9)
    ax.set_ylabel('FLUXCAL', fontsize=9)
    ax.legend(fontsize=6, ncol=6, loc='upper right')
    ax.set_title(
        f"SNID {snid}  z={z_true:.3f}  "
        f"x1={res['x1']:+.2f}±{pe.get('x1',float('nan')):.2f}  "
        f"c={res['c']:+.2f}±{pe.get('c',float('nan')):.2f}  "
        f"χ²/dof={res['chi2_red']:.2f}",
        fontsize=9
    )


print("plot_lightcurve ready.")

In [ ]:
# ── Figure 2 : corner plot of pairwise error ellipses ─────────────────────────

def plot_corner_ellipses(res: dict, snid: int, z_true: float,
                         param_names=None) -> plt.Figure:
    """Draw a corner / scatter-matrix plot with pairwise 1σ & 2σ error ellipses.

    The diagonal panels show the 1D Gaussian marginal ± 1σ.
    The off-diagonal lower-triangle panels show the 2D confidence ellipses.
    The upper-triangle panels show the Pearson correlation coefficient ρ_ij.

    Parameters
    ----------
    res         : dict returned by fit_salt3_event
    snid        : object identifier for the title
    z_true      : truth redshift for the title
    param_names : list of parameter names to include (default: VPARAM_NAMES)

    Returns
    -------
    matplotlib Figure
    """
    if param_names is None:
        param_names = VPARAM_NAMES

    vcov = res.get('vcov')
    corr = res.get('corr_matrix')
    pe   = res.get('param_errors', {})
    vp   = res.get('vparam_names', param_names)

    # Index of each desired param in the fitted vparam_names list
    try:
        indices = [vp.index(p) for p in param_names]
    except ValueError as e:
        raise ValueError(f"Parameter not found in fit result: {e}")

    n = len(param_names)
    best = {p: float(res[p]) for p in param_names}

    fig, axes = plt.subplots(n, n, figsize=(3.2 * n, 3.2 * n),
                             tight_layout=True)

    ELLIPSE_COLOR = '#1f77b4'

    for row in range(n):
        pi   = param_names[row]
        si   = pe.get(pi, 0.0)
        ci   = best[pi]
        ii   = indices[row]

        for col in range(n):
            ax  = axes[row, col]
            pj  = param_names[col]
            sj  = pe.get(pj, 0.0)
            cj  = best[pj]
            jj  = indices[col]

            # ── Diagonal: 1D Gaussian marginal ─────────────────────────────────
            if row == col:
                if si > 0:
                    xg = np.linspace(ci - 4 * si, ci + 4 * si, 300)
                    yg = np.exp(-0.5 * ((xg - ci) / si) ** 2) / (si * np.sqrt(2 * np.pi))
                    ax.plot(xg, yg, color=ELLIPSE_COLOR, lw=2)
                    ax.axvline(ci,      color='k',   lw=1.2, ls='-')
                    ax.axvline(ci + si, color='gray', lw=0.8, ls='--')
                    ax.axvline(ci - si, color='gray', lw=0.8, ls='--')
                    # Shade ±1σ
                    xfill = xg[(xg >= ci - si) & (xg <= ci + si)]
                    yfill = np.exp(-0.5 * ((xfill - ci) / si) ** 2) / (si * np.sqrt(2 * np.pi))
                    ax.fill_between(xfill, yfill, alpha=0.25, color=ELLIPSE_COLOR)
                ax.set_xlim(ci - 4 * max(si, 1e-12), ci + 4 * max(si, 1e-12))
                ax.set_yticks([])
                ax.set_xlabel(PARAM_LABELS.get(pi, pi), fontsize=9)
                # Value annotation
                ax.set_title(
                    f"{pi} = {ci:.4g}\n± {si:.3g}",
                    fontsize=8
                )

            # ── Lower triangle: 2D error ellipses ─────────────────────────────
            elif row > col:
                # Horizontal axis = pj (col), vertical axis = pi (row)
                if vcov is not None:
                    # Extract 2×2 sub-covariance [pj, pi] order (x=col, y=row)
                    cov2 = np.array([
                        [vcov[jj, jj], vcov[jj, ii]],
                        [vcov[ii, jj], vcov[ii, ii]]
                    ])
                    add_error_ellipse(ax, cj, ci, cov2,
                                      color=ELLIPSE_COLOR,
                                      alpha1=0.35, alpha2=0.15)

                # Best-fit point
                ax.plot(cj, ci, '+', color='k', ms=8, mew=1.5, zorder=5)

                # Axis limits: ±3σ around best fit
                ax.set_xlim(cj - 3 * max(sj, 1e-12), cj + 3 * max(sj, 1e-12))
                ax.set_ylim(ci - 3 * max(si, 1e-12), ci + 3 * max(si, 1e-12))
                ax.set_xlabel(PARAM_LABELS.get(pj, pj), fontsize=9)
                ax.set_ylabel(PARAM_LABELS.get(pi, pi), fontsize=9)
                ax.tick_params(labelsize=7)

                # Legend for 1σ / 2σ only on bottom-left panel
                if row == n - 1 and col == 0:
                    p1 = mpatches.Patch(color=ELLIPSE_COLOR, alpha=0.35,
                                        label=r'1$\sigma$ (Δχ²=2.30)')
                    p2 = mpatches.Patch(color=ELLIPSE_COLOR, alpha=0.15,
                                        label=r'2$\sigma$ (Δχ²=6.18)')
                    ax.legend(handles=[p1, p2], fontsize=7, loc='upper left')

            # ── Upper triangle: correlation coefficient ρ_ij ──────────────────
            else:
                ax.set_xlim(0, 1)
                ax.set_ylim(0, 1)
                ax.set_xticks([])
                ax.set_yticks([])
                if corr is not None:
                    rho = corr[ii, jj]
                    if np.isfinite(rho):
                        # Colour-code: red=positive, blue=negative
                        cmap = plt.cm.RdBu_r
                        rgba = cmap(0.5 * (rho + 1.0))
                        ax.set_facecolor(rgba)
                        ax.text(0.5, 0.5,
                                f"ρ = {rho:+.3f}",
                                ha='center', va='center',
                                fontsize=11, fontweight='bold',
                                color='white' if abs(rho) > 0.5 else 'black',
                                transform=ax.transAxes)
                ax.set_xlabel(PARAM_LABELS.get(pj, pj), fontsize=9)
                ax.set_ylabel(PARAM_LABELS.get(pi, pi), fontsize=9)

    fig.suptitle(
        f"SALT3 error ellipses — SNID {snid}  z={z_true:.3f}  "
        f"χ²/dof={res['chi2_red']:.2f}\n"
        r"Lower triangle: 1$\sigma$ (filled) + 2$\sigma$ (dashed) ellipses  |  "
        r"Upper triangle: Pearson $\rho_{ij}$",
        fontsize=10, y=1.02
    )
    return fig


print("plot_corner_ellipses ready.")

In [ ]:
# ── Figure 3 : parameter correlation matrix heatmap ───────────────────────────

def plot_correlation_matrix(res: dict, snid: int, z_true: float,
                             param_names=None) -> plt.Figure:
    """Draw the full Pearson correlation matrix as an annotated heatmap.

    Parameters
    ----------
    res         : dict returned by fit_salt3_event
    snid        : object identifier
    z_true      : truth redshift
    param_names : list of parameter names (default: VPARAM_NAMES)

    Returns
    -------
    matplotlib Figure
    """
    if param_names is None:
        param_names = VPARAM_NAMES

    corr = res.get('corr_matrix')
    vp   = res.get('vparam_names', param_names)

    try:
        indices = [vp.index(p) for p in param_names]
    except ValueError as e:
        raise ValueError(f"Parameter not found: {e}")

    n = len(param_names)

    # Extract the relevant n×n sub-matrix in the order of param_names
    sub_corr = np.array([[corr[i, j] for j in indices] for i in indices])

    fig, ax = plt.subplots(figsize=(0.9 * n + 2.5, 0.9 * n + 2.0),
                            tight_layout=True)

    im = ax.imshow(sub_corr, vmin=-1, vmax=1, cmap='RdBu_r', aspect='equal')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label='Pearson correlation coefficient ρ')

    tick_labels = [PARAM_LABELS.get(p, p) for p in param_names]
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(tick_labels, fontsize=11)
    ax.set_yticklabels(tick_labels, fontsize=11)
    ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

    # Annotate each cell with the correlation value
    for i in range(n):
        for j in range(n):
            val = sub_corr[i, j]
            text_color = 'white' if abs(val) > 0.65 else 'black'
            ax.text(j, i, f"{val:+.3f}",
                    ha='center', va='center',
                    fontsize=11, fontweight='bold',
                    color=text_color)

    # Draw grid lines between cells
    for k in range(n + 1):
        ax.axhline(k - 0.5, color='white', lw=0.5)
        ax.axvline(k - 0.5, color='white', lw=0.5)

    ax.set_title(
        f"SALT3 parameter correlation matrix\n"
        f"SNID {snid}  z={z_true:.3f}  χ²/dof={res['chi2_red']:.2f}",
        fontsize=11, pad=16
    )
    return fig


print("plot_correlation_matrix ready.")

## 4 · Select events and run SALT3 fits

In [ ]:
n_avail      = min(N_CURVES, len(subset))
chosen_idx   = rng.choice(len(subset), size=n_avail, replace=False)
chosen_snids = subset['SNID'].values[chosen_idx]
print(f"Selected {n_avail} SNIDs.")

In [ ]:
fit_results = {}

for snid in chosen_snids:
    ltcv   = all_ltcvs[all_ltcvs['SNID'] == snid]
    z_row  = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan

    res = fit_salt3_event(ltcv, z_true=z_true,
                          fit_z=False,
                          photflag_detect=esr.photflag_detect)
    fit_results[snid] = res

    if res['success']:
        pe = res['param_errors']
        print(
            f"  SNID {snid:8d}  ✓  z={res['z']:.4f}  "
            f"t0={res['t0']:.2f}±{pe.get('t0',float('nan')):.2f}  "
            f"x0={res['x0']:.3e}±{pe.get('x0',float('nan')):.2e}  "
            f"x1={res['x1']:+.3f}±{pe.get('x1',float('nan')):.3f}  "
            f"c={res['c']:+.3f}±{pe.get('c',float('nan')):.3f}  "
            f"χ²/dof={res['chi2_red']:.2f}"
        )
    else:
        print(f"  SNID {snid:8d}  ✗  {res['message']}")

## 5 · Per-event diagnostic figures

For each event the following three figures are produced:

- **Figure A** — multi-band light curve + SALT3 model
- **Figure B** — corner plot: pairwise 1σ/2σ error ellipses + ρ values
- **Figure C** — full correlation matrix heatmap

In [ ]:
for snid in chosen_snids:
    res    = fit_results[snid]
    z_row  = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan

    print("\n" + "═" * 70)
    print(f"  SNID {snid}   z_true = {z_true:.4f}")
    print("═" * 70)

    # ── Figure A : light curve ────────────────────────────────────────────────
    fig_lc, ax_lc = plt.subplots(figsize=(10, 4), tight_layout=True)
    plot_lightcurve(ax_lc, snid, res, z_true)
    fig_lc.suptitle(
        f"Figure A — SALT3 light-curve fit   SNID {snid}   "
        f"Model: {SALT3_SOURCE} v{SALT3_VERSION}",
        fontsize=10, y=1.02
    )
    plt.tight_layout()
    plt.show()

    if not res.get('success') or res.get('vcov') is None:
        print(f"  No covariance available for SNID {snid} — skipping B & C.")
        continue

    # ── Figure B : corner plot with error ellipses ────────────────────────────
    fig_corner = plot_corner_ellipses(res, snid, z_true)
    plt.show()

    # ── Figure C : correlation matrix heatmap ─────────────────────────────────
    fig_corr = plot_correlation_matrix(res, snid, z_true)
    plt.show()

    # ── Print the numerical covariance and correlation matrices ───────────────
    vp  = res['vparam_names']
    pe  = res['param_errors']
    vcov = res['vcov']
    corr = res['corr_matrix']

    print("\n  Best-fit parameters and 1σ uncertainties:")
    for p in VPARAM_NAMES:
        val = res.get(p, float('nan'))
        err = pe.get(p, float('nan'))
        print(f"    {p:3s}  =  {val:+.5g}  ±  {err:.5g}")

    print("\n  Covariance matrix C (rows/cols: t0, x0, x1, c):")
    idx = [vp.index(p) for p in VPARAM_NAMES]
    sub_cov  = vcov[np.ix_(idx, idx)]
    sub_corr = corr[np.ix_(idx, idx)]
    header   = '         ' + ''.join(f"  {p:>10s}" for p in VPARAM_NAMES)
    print(header)
    for i, pi in enumerate(VPARAM_NAMES):
        row_str = f"    {pi:3s}  " + ''.join(f"  {sub_cov[i,j]:+10.4e}" for j in range(len(VPARAM_NAMES)))
        print(row_str)

    print("\n  Correlation matrix ρ (rows/cols: t0, x0, x1, c):")
    print(header)
    for i, pi in enumerate(VPARAM_NAMES):
        row_str = f"    {pi:3s}  " + ''.join(f"  {sub_corr[i,j]:+10.4f}" for j in range(len(VPARAM_NAMES)))
        print(row_str)

## 6 · Aggregate view: correlation matrices for all fitted events

Compact grid of correlation matrix heatmaps — one per event — to survey the
spread of inter-parameter correlations across the sample.

In [ ]:
good_snids = [snid for snid in chosen_snids
              if fit_results[snid].get('success') and
                 fit_results[snid].get('corr_matrix') is not None]

n_good = len(good_snids)
NCOLS_AGG = min(4, n_good)
nrows_agg = math.ceil(n_good / NCOLS_AGG)

fig_agg, axes_agg = plt.subplots(
    nrows_agg, NCOLS_AGG,
    figsize=(3.2 * NCOLS_AGG, 3.0 * nrows_agg),
    tight_layout=True
)
axes_agg_flat = np.array(axes_agg).flatten()

tick_labels = [PARAM_LABELS.get(p, p) for p in VPARAM_NAMES]
n_p = len(VPARAM_NAMES)

for idx, snid in enumerate(good_snids):
    ax    = axes_agg_flat[idx]
    res   = fit_results[snid]
    vp    = res['vparam_names']
    corr  = res['corr_matrix']
    z_row = truth[truth['SNID'] == snid]['ZCMB'].values
    z_true = float(z_row[0]) if len(z_row) else np.nan

    pidx     = [vp.index(p) for p in VPARAM_NAMES]
    sub_corr = corr[np.ix_(pidx, pidx)]

    im = ax.imshow(sub_corr, vmin=-1, vmax=1, cmap='RdBu_r', aspect='equal')

    ax.set_xticks(range(n_p))
    ax.set_yticks(range(n_p))
    ax.set_xticklabels(tick_labels, fontsize=7, rotation=45, ha='right')
    ax.set_yticklabels(tick_labels, fontsize=7)

    for i in range(n_p):
        for j in range(n_p):
            val = sub_corr[i, j]
            ax.text(j, i, f"{val:+.2f}",
                    ha='center', va='center', fontsize=6.5,
                    color='white' if abs(val) > 0.65 else 'black')

    ax.set_title(f"SNID {snid}\nz={z_true:.3f}  χ²/dof={res['chi2_red']:.2f}",
                 fontsize=7)

# Hide unused panels and add common colorbar
for idx in range(n_good, len(axes_agg_flat)):
    axes_agg_flat[idx].set_visible(False)

# Common colorbar on the right
cbar = fig_agg.colorbar(
    im, ax=axes_agg_flat[:n_good],
    fraction=0.02, pad=0.04,
    label='ρ'
)
cbar.ax.tick_params(labelsize=8)

fig_agg.suptitle(
    f"SALT3 correlation matrices — {OBJ_CLASS}  "
    f"({n_good} events,  params: {', '.join(VPARAM_NAMES)})",
    fontsize=11, y=1.02
)
plt.tight_layout()
plt.show()

## 7 · Distribution of pairwise correlation coefficients across the sample

For each parameter pair $(p_i, p_j)$ with $i < j$ we collect $\rho_{ij}$ over all
successfully fitted events and show its distribution as a histogram.

In [ ]:
from itertools import combinations

pairs = list(combinations(range(len(VPARAM_NAMES)), 2))
n_pairs = len(pairs)

# Collect ρ values per pair
rho_collections = {(i, j): [] for i, j in pairs}

for snid in good_snids:
    res  = fit_results[snid]
    vp   = res['vparam_names']
    corr = res['corr_matrix']
    pidx = [vp.index(p) for p in VPARAM_NAMES]
    sub  = corr[np.ix_(pidx, pidx)]
    for (i, j) in pairs:
        val = sub[i, j]
        if np.isfinite(val):
            rho_collections[(i, j)].append(val)

fig_rho, axes_rho = plt.subplots(
    1, n_pairs, figsize=(4.2 * n_pairs, 3.5), tight_layout=True
)

for ax, (i, j) in zip(axes_rho, pairs):
    pi, pj = VPARAM_NAMES[i], VPARAM_NAMES[j]
    vals = rho_collections[(i, j)]
    if len(vals) == 0:
        ax.set_title(f'ρ({pi},{pj})\nno data', fontsize=9)
        continue
    ax.hist(vals, bins=max(5, len(vals) // 3), color='steelblue', edgecolor='white')
    med = np.median(vals)
    ax.axvline(med, color='k', ls='--', lw=1.5, label=f'median={med:+.3f}')
    ax.axvline(0.0, color='r', ls=':', lw=1.0)
    ax.set_xlabel(
        rf'$\rho$({PARAM_LABELS.get(pi, pi)}, {PARAM_LABELS.get(pj, pj)})',
        fontsize=10
    )
    ax.set_ylabel('N events', fontsize=10)
    ax.set_title(f'σ = {np.std(vals):.3g}', fontsize=9)
    ax.set_xlim(-1, 1)
    ax.legend(fontsize=8)

fig_rho.suptitle(
    f"Distribution of pairwise ρ — {OBJ_CLASS}  ({len(good_snids)} events)",
    fontsize=11
)
plt.show()

## Summary

### Figures produced per event

| Figure | Content |
|--------|---------|
| **A** | Multi-band FLUXCAL light curve + SALT3 model, colour-coded by band |
| **B** | Corner plot: diagonal = 1D Gaussian marginal; lower triangle = 1σ & 2σ error ellipses from covariance; upper triangle = Pearson ρ |
| **C** | Full 4×4 correlation matrix heatmap with annotated ρ values |

### Error ellipse construction

The 2D ellipses are derived analytically from the **4×4 covariance matrix** returned
by `sncosmo.fit_lc` via eigendecomposition of the 2×2 sub-matrix for each parameter
pair $(p_i, p_j)$.

| Confidence level | Δχ² (2 d.o.f.) | Interpretation |
|---|---|---|
| 1σ | 2.30 | 68.3 % joint probability |
| 2σ | 6.18 | 95.4 % joint probability |

### Typical correlations in SALT3 fits

- **x1 – c** : often small but non-zero; degenerate if few blue-band observations.
- **x0 – x1** : negative; a brighter event (large x0) can partially mimic a wider
  light curve (x1 > 0) if wavelength coverage is limited.
- **t0 – x0** : typically small; larger if the rise phase is poorly sampled.
- **t0 – x1** : moderate; stretch shifts the time of effective peak flux slightly.

These correlations are the main source of systematic uncertainty when using
SALT3 parameters for cosmological distance estimation.
